[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-biology/AdvancesMSDataProcessing/blob/master/MultivariateExploratoryAnalysisSupervisedModelsClassificationPrediction/Stats_Untargeted_Metabolomics_python.ipynb)


# Performing basic uni- and multivariate statistical analsysis of untargeted metabolomics data

**Updated on:** 2024-02-17 17:33 CET

In this Jupyter Notebook we perform basic explorative uni- and multivariate statistical analyses of an untargeted metabolomics data set, including data cleaning steps, normalization and batch correction. **All step numbers in the Python Notebook correspond to those in the R Notebook to ensure consistency and comparability with the main protocol, which primarily uses the R Notebook.**

<div class="alert alert-block alert-info">

**Authors**: Abzer Kelminal (abzer.shah@uni-tuebingen.de), Francesco Russo (frru@ssi.dk), Filip Ottosson (faot@ssi.dk), Madaleine Ernst (maet@ssi.dk), Axel Walter (axel.walter@uni-tuebingen.de), Carolina Gonzalez (cgonzalez7@eafit.edu.co), Efi Kontou (eeko@biosustain.dtu.dk), Judith Boldt (judith.boldt@dsmz.de) <br>

**Input file format**: .csv files or .txt files <br>
**Outputs**: .csv files, .pdf & .svg images  <br>
**Dependencies**: pandas, numpy, glob, os, re, itertools, plotly, matplotlib, scipy, sklearn, pingouin, skbio, ipyfilechooser, ipywidgets, pynmranalysis, PyComplexHeatmap

The session info at the end of this notebook gives info about the versions of all the packages used here.
</div>

---
<font color = 'red' size =4>**Disclaimer**: This Notebook is compatible with both Jupyter Notebook and Google Colab. However, we advise users with a Windows OS to opt for the Colab version due to potential compatibility issues with certain modules when installed locally on Windows. </font>

---

<b> Before starting to run this notebook with your own data, remember to save a copy of this notebook in your own Google Drive! Do so by clicking on File &rarr; Save a copy in Drive. You can give whatever meaningful name to your notebook.
This file should be located in a new folder of your Google Drive named 'Colab Notebooks'. You can also download this notebook: File &rarr; Download &rarr; Download .ipynb.</b>

---

<b><font color='red' size=4> Please read the comments before proceeding with the code and let us know if you run into any errors and if you think it could be commented better. We would highly appreciate your suggestions and comments!!</font> </b>

# <font color ='blue'> 1. Introduction </font>
<a id='intro'></a>

<p style='text-align: justify;'> The example files used in this tutorial are part of a study published by <a href="https://doi.org/10.1016/j.chemosphere.2020.129450">Petras and coworkers (2021)</a>. Here, the authors investigated the coastal environments in northern San Diego, USA, after a major rainfall event in winter 2017/2018 to observe the seawater chemotype. The dataset contains surface seawater samples collected (−10 cm) at 30 sites spaced approximately 300 meters apart and 50–100 meters offshore along the San Diego coastline from Torrey Pines State Beach to Mission Bay (Torrey Pines, La Jolla Shores, La Jolla Reefs, Pacific and Mission Beach) at 3 different time points: Dec 2017, Jan 2018 (after a major rainfall, resulting in decreased salinity of water) and Oct 2018. As a result of the study, a huge shift was observed in the seawater's organic matter chemotype after the rainfall. <br>

<p style='text-align: justify;'> Seawater samples collected during October 2018, were not published in the original article, but are added to this tutorial to have increased sample size. The datasets used here can be found in the MassIVE repository: <a href="https://massive.ucsd.edu/ProteoSAFe/dataset.jsp?task=8a8139d9248b43e0b0fda17495387756">MSV000082312</a> and <a href="https://massive.ucsd.edu/ProteoSAFe/dataset.jsp?task=c8411b76f30a4f4ca5d3e42ec13998dc">MSV000085786</a>. The .mzml files
were preprocessed using <a href="http://mzmine.github.io/">MZmine3</a> and the <a href="https://gnps.ucsd.edu/ProteoSAFe/status.jsp?task=b661d12ba88745639664988329c1363e">feature-based molecular networking workflow in GNPS</a>.</p>

<p style='text-align: justify;'> We initially clean the feature table by batch correction, blank removal, imputation, normalization, and scaling. Then we perform multivariate and univariate statistical analyses including supervised classification analysis (Random Forest) and unsupervised learning methods (e.g., PCoA and clustering). Each step is discussed briefly in its own sections.</p>

---

### General Instructions

- **<font color='red'>Text in Red:</font>** This color is used to highlight critical information or indicate cells that require user input. Please pay close attention to these instructions. These instructions guide the user to adapt the notebook for their own datasets without having to modify the code. Cells requiring input will not progress without the user's response. Further details are provided within the notebook itself.

- **User Prompt Guidance:** When encountering code cells subsequent to the highlighted red text, make sure to just run the cell without modifying the code. For example, when you see a code like this:
`data_dir = input("Enter the path of the folder with input files:\n")`. For such prompts, run the code first and provide your answers in the output cell that appears after running the code, not directly within the code itself. This ensures that user input is seamlessly integrated into the subsequent operations.

- **<font color='green'>Text in Green:</font>** Cells with this color contain function blocks. These cells will not produce any visible outputs when run, as they are defining functions for later use in the notebook.

- **Using the '#' Operator:** In the code cells, you may notice lines starting with the '#' symbol. These are comments that describe the function or purpose of a code section. Lines beginning with '#' are "commented out" and will not execute when the cell is run. If you wish to run a commented-out code, simply remove the '#' symbol at the beginning of the line and run the cell again.
---

# <font color ='blue'> 2. Preliminary setup for the notebook </font>
<a id='Section-2'>

### <font color ='darkblue'> Step 1: Choose a Notebook and install it. </font>

We recommend beginners use Google Colab for the Python notebook due to its hassle-free setup as it requires no installations, making it accessible for those unfamiliar with the setup process. However, for regular analysis, local execution in Jupyter on a contemporary desktop computer (E.g., Intel i7, 16 core, 64 GB RAM) is typically faster and more efficient.

Windows users are advised to use Anaconda Navigator for a straightforward setup of Jupyter Notebook, with detailed instructions provided in our guide. Extensive documentation for Anaconda Navigator can also be found at Anaconda’s official site (https://docs.anaconda.com/free/navigator/). Mac OS users may encounter difficulties with Anaconda Navigator; thus, we suggest the Homebrew-based installation.

## Package installation

### <font color ='darkblue'> Step 2: Installing and loading necessary packages </font>
<a id = 'pkg_install'></a>
Before we start, we need to install all packages needed for our analyses and load the installed packages into our session.

In [ ]:
# Install libraries that are not preinstalled
!pip install pandas numpy plotly scikit-learn scikit-posthocs pingouin kaleido nbformat session_info PyComplexHeatmap yellowbrick matplotlib

In [ ]:
# importing necessary modules
import pandas as pd
import numpy as np

import glob
import os
import re
import itertools
import io
import session_info
import shutil
import warnings
import matplotlib.pyplot as plt
import platform
import math
import scipy.stats as stats

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.io as pio

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import fcluster
from scipy.spatial import distance
from scipy.stats import lognorm
from scipy.stats import shapiro
from scipy.stats import lognorm
from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import class_weight
from sklearn.metrics import classification_report
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

import pingouin as pg
import scikit_posthocs as sp
from PyComplexHeatmap import *
from yellowbrick.cluster import KElbowVisualizer

from PIL import Image

import statsmodels.api as sm

import time
from sklearn.inspection import permutation_importance

In [ ]:
# Get the operating system information
os_info = platform.system()

# Print the operating system
print(f"The operating system is: {os_info}")

The operating system is: Linux


<font color='red' size= 4 > Installing the following module is not a problem in Google Colab (as it is Linux-based) as well as in macOS and Linux-based systems.</font> scikit-bio is not supported for Windows OS.

In [ ]:
if os != "Windows":
    !pip install scikit-bio
    import skbio
    from skbio.stats.distance import permdisp
    print('You can compute PCoA and PERMANOVA')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.7 MB/s eta 0:00:00
You can compute PCoA and PERMANOVA


In [ ]:
# Disable warnings for cleaner output, comment out for debugging
warnings.filterwarnings('ignore')

### <font color ='darkblue'> Setting a local data directory </font>
<a name = "set_dir"></a>

<p style='text-align: justify;'> When we set a folder (or directory) as the data directory, we can access the files within the folder more easily without mentioning the entire file path everytime we use it. Also, all the output files will be saved under the data directory. So, before proceeding with the script further, if you are trying to use your own files for the notebook, then please make sure to include all the necessary input files in one local folder and set it as your data directory. <br/>
    
<b> Uploading Files in Google Colab:</b>
1. In the Google Colab homepage, notice the three icons in the upper left corner.
2. Click on the three dots icon for a detailed view of your notebook's contents.
3. To create a folder for your input files:
   - Click on the folder icon. Note the default 'sample_data' folder present in Google Colab.
   - Right-click in an empty space within the left panel of Google Colab.
   - Choose 'new folder' from the dropdown.
   - Copy the path of the folder you just created.
   - Copy the path and paste in the output box of your next cell for precise referencing.

<b>Note for clarity</b>: Folders you create within Colab default to the "/content/" directory. This is the same directory housing the 'sample_data' folder.

<font color='red' size=4> Run the cell below and enter the path of the folder containing all your input files in the output cell.</font> <br/>
For example:  
- D:\User\Project\test_data (in case of running the notebook locally)  
- /content/test_data (in Google Colab)

<font color='red' size=4> Whenever you see an output box asking for user input, please note, the script will not proceed further without your input. Hence, make sure to provide the necessary input and press 'Enter' to move too the next cell.Also, run the notebook cell-by-cell instead of running all cells in the notebook simultaneously. </font>

---
## Data import
<a name='load_ip'></a>

<b><u>Files needed for running the Notebook:</b></u>  

1) <b>Feature table:</b> A typical output file of an LC-MS/MS metabolomics experiment, containing all mass spectral features (or peaks) with their corresponding relative intensities across samples. The feature table we use in this tutorial was obtained from MZmine3. (Filetype: .csv file) <br>

2) <b>Metadata:</b> An Excel file saved in .txt format that is created by the user, providing additional information about the samples (e.g. sample type, tissue type, species, timepoint of collection etc.) In this tutorial we are using the [metadata format recognized by GNPS workflows](https://ccms-ucsd.github.io/GNPSDocumentation/metadata/). The first column should be named 'filename' and all remaining column headers should be prefixed with ATTRIBUTE_: e.g. ATTRIBUTE_Sample_Type, ATTRIBUTE_timepoint etc. (Filetype: .txt file) <br>

Feature table and metadata used in this tutorial can be accessed at <a href="https://github.com/Functional-Metabolomics-Lab/FBMN-STATS/tree/main/data">our Functional Metabolomics Git Repository.</a>

3) <b>OPTIONAL:</b> Annotation files such as GNPS library annotations, GNPS analog annotations, SIRIUS annotations.

<b><u>More on Annotation files:</u></b>  
<p style='text-align: justify;'> If available, provide the files for molecular annotations such as SIRIUS, CANOPUS annotation files. SIRIUS and CANOPUS performs molecular formula prediction and chemical class predictions respectively. GNPS annotation files are obtained as a result of Feature-Based Molecular Networking (FBMN) analysis on the feature table (provided along with its corresponding metadata). <a href="https://gnps.ucsd.edu/ProteoSAFe/status.jsp?task=b661d12ba88745639664988329c1363e">The FBMN job is also publicly available.</a> These GNPS annotation files can be directly accessed with their task ID. More on this on <a href='#gnps_files'>Step 5C</a>.</p>

<b><u>Downloading FBMN Results Locally:</u></b>

1. For those using a **MassIVE** or **GNPS** account:
   - Navigate to "Jobs".
   - Click on the **Status** of your FBMN Workflow.
   - If accessing via a direct link to the Status page, proceed to the next step.
2. Click "Download Cytoscape Data" to obtain a folder named, for example, "ProteoSAFe-FEATURE-BASED-MOLECULAR-NETWORKING-5877234d-download_cytoscape_data".
3. After downloading, unzip the folder. Inside, you will find sub-folders with files such as:
   - 'quantification_table' (contains the Feature table)
   - 'metadata_table' (contains the Metadata table)
   - 'gnps_molecular_network_graphml' (for Molecular network visualization).
4. For annotations on compounds:
   - Refer to the **DB_result** folder for the GNPS spectral library search.
   - If an advanced analog search was performed, see the **DB_analog_result** folder.

Note: The analog search searches for structurally related molecules within the molecular network using a score threshold, such as a minimum cosine score that MS/MS spectra should achieve in spectral matching with MS/MS spectral libraries to be considered an annotation. As an example, our GNPS annotation found 255 compounds, while the analog search detected 1991. It is crucial to verify analog annotations by examining both the cosine score and mirror match accuracy.

[![More on MZmine](https://img.shields.io/badge/More%20on-MZmine-blue)](https://www.nature.com/articles/s41587-023-01690-2)
[![More on GNPS](https://img.shields.io/badge/More%20on-GNPS-informational)](https://www.nature.com/articles/nbt.3597#Abs2)
[![More on FBMN](https://img.shields.io/badge/More%20on-FBMN-blue)](https://www.nature.com/articles/s41592-020-0933-6)
[![More on SIRIUS](https://img.shields.io/badge/More%20on-SIRIUS-blue)](https://boecker-lab.github.io/docs.sirius.github.io/)

---

### Loading files from a local folder

Please make sure to include all the necessary input files in the folder you have set as input directory

In [ ]:
#importing data directly from github
ft = pd.read_csv('https://raw.githubusercontent.com/Functional-Metabolomics-Lab/FBMN-STATS/refs/heads/main/data/SD_BeachSurvey_GapFilled_quant.csv')
md = pd.read_csv('https://raw.githubusercontent.com/Functional-Metabolomics-Lab/FBMN-STATS/refs/heads/main/data/20221125_Metadata_SD_Beaches_with_injection_order.txt', sep = "\t")

---
<font color="red" size=4> In the cell below, provide the index number of the annotation files. Be careful to input the numbers without any spaces. </font>

In [ ]:
an_gnps = pd.read_csv('https://raw.githubusercontent.com/Functional-Metabolomics-Lab/FBMN-STATS/refs/heads/main/data/GNPS_result_FBMN.tsv', sep = "\t")
an_analog = pd.read_csv('https://raw.githubusercontent.com/Functional-Metabolomics-Lab/FBMN-STATS/refs/heads/main/data/GNPS_analog_result_FBMN.tsv', sep = "\t")


---
<font color="red" size=4> If you also have SIRIUS annotation file, provide the index number of the file in the cell below.</font>

Once the files are loaded, you can view the files as shown in <a href='#view_ip'>Step 6.</a>

### <font color ='darkblue'>  Exploring the Imported Files </font>
<a id='view_ip'></a>

Lets check how the data looks, the below lines of code show the first 5 rows of the feature and metadata tables, and the first 2 rows of the annotation tables, as well as the dimensions of all tables (numbers of rows and columns)

In [ ]:
print('Dimension: ', ft.shape) #gets the dimension (number of rows and columns) of ft
ft.head() # gets the first 5 rows of ft

Dimension:  (11217, 200)


,row ID,row m/z,row retention time,row ion mobility,row ion mobility unit,row CCS,correlation group ID,annotation network number,best ion,auto MS2 verify,...,SD_12-2017_15_b.mzXML Peak area,SD_12-2017_15_a.mzXML Peak area,SD_12-2017_27_a.mzXML Peak area,SD_12-2017_29_b.mzXML Peak area,SD_12-2017_21_a.mzXML Peak area,SD_12-2017_30_a.mzXML Peak area,SD_12-2017_28_b.mzXML Peak area,SD_12-2017_29_a.mzXML Peak area,SD_12-2017_28_a.mzXML Peak area,Unnamed: 199
0,92572,151.035101,13.363672,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,21385.480,1.138271e+03,1144.8115,12139.16,5.394689e+03,5.270766e+03,1.007839e+03,NaN
1,2513,151.035125,1.129901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,27123.893,0.000000e+00,0.0000,0.00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
2,42,151.035140,0.550724,NaN,NaN,NaN,212.0,NaN,NaN,NaN,...,1150350.0,1103477.9,2638109.200,1.446267e+06,595216.5000,1225695.20,1.424855e+06,1.557217e+06,1.797692e+06,NaN
3,1870,151.035199,0.886780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,314371.840,0.000000e+00,0.0000,0.00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
4,2127,151.096405,0.986017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.000,0.000000e+00,0.0000,0.00,0.000000e+00,0.000000e+00,0.000000e+00,NaN


In [ ]:
print('Dimension: ', md.shape)
md.head()

Dimension:  (186, 14)


,filename,ATTRIBUTE_Sample_Type,ATTRIBUTE_Batch,ATTRIBUTE_Month,ATTRIBUTE_Year,ATTRIBUTE_Sample_Location,ATTRIBUTE_Replicate,ATTRIBUTE_Spot,ATTRIBUTE_Latitude,ATTRIBUTE_Longitude,ATTRIBUTE_Sample_Area,ATTRIBUTE_Spot_Name,ATTRIBUTE_time_run,ATTRIBUTE_Injection_order
0,SD_10_2018_10_a.mzXML,Sample,3,Oct,2018,10,a,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,18/07/2020 18:19,145
1,SD_10_2018_10_b.mzXML,Sample,3,Oct,2018,10,b,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,18/07/2020 18:35,146
2,SD_10_2018_11_a.mzXML,Sample,3,Oct,2018,11,a,11,32.85601,-117.26253,SIO_La_Jolla_Shores,La_Jolla_Shores,18/07/2020 18:51,147
3,SD_10_2018_11_b.mzXML,Sample,3,Oct,2018,11,b,11,32.85601,-117.26253,SIO_La_Jolla_Shores,La_Jolla_Shores,18/07/2020 19:07,148
4,SD_10_2018_12_a.mzXML,Sample,3,Oct,2018,12,a,12,32.85161,-117.26965,La_Jolla_Cove,Cove,18/07/2020 19:23,149


In [ ]:
an_gnps.head(n=2)

,SpectrumID,Compound_Name,Ion_Source,Instrument,Compound_Source,PI,Data_Collector,Adduct,Precursor_MZ,ExactMass,...,MoleculeExplorerDatasets,MoleculeExplorerFiles,InChIKey,InChIKey-Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway
0,CCMSLIB00005466086,Irgarol,LC-ESI,Orbitrap,Commercial,Daniel Petras,Daniel Petras,M+H,254.143,254.143,...,0,0,HDHLIWCXDDZUFH-UHFFFAOYSA-N,HDHLIWCXDDZUFH,NaN,NaN,NaN,NaN,NaN,Alkaloids
1,CCMSLIB00003136915,Spectral Match to N-Lauroylsarcosine from NIST14,ESI,QqQ,Isolated,Data from Nediljko Budisa,Data deposited by daniel,M+H,272.222,0.000,...,60,3116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
an_analog.head(n=2)

,SpectrumID,Compound_Name,Ion_Source,Instrument,Compound_Source,PI,Data_Collector,Adduct,Precursor_MZ,ExactMass,...,MoleculeExplorerDatasets,MoleculeExplorerFiles,InChIKey,InChIKey-Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway
0,CCMSLIB00005466086,Irgarol,LC-ESI,Orbitrap,Commercial,Daniel Petras,Daniel Petras,M+H,254.143,254.143,...,0,0,HDHLIWCXDDZUFH-UHFFFAOYSA-N,HDHLIWCXDDZUFH,NaN,NaN,NaN,NaN,NaN,Alkaloids
1,CCMSLIB00003136915,Spectral Match to N-Lauroylsarcosine from NIST14,ESI,QqQ,Isolated,Data from Nediljko Budisa,Data deposited by daniel,M+H,272.222,0.000,...,60,3116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print('Dimension: ', an_gnps.shape)
print('Dimension: ', an_analog.shape)

Dimension:  (260, 46)
Dimension:  (1991, 46)


Out of the 11217 features, we found 260 hits using GNPS library and around 2000 hits using analog search.

---
### <font color ='darkblue'>  Summarizing the metadata </font>
<a id='explore_md'></a>

<p style='text-align: justify;'>Before starting with our analysis, we take a look at our metadata. For this purpose, we have created a function. A function is a collection of commands, which takes one or multiple input variables and creates a corresponding output. By creating functions, we avoid having to write big code chunks multiple times. Instead, we can call a sequence of code lines by their function name.</p>
    
<p style='text-align: justify;'><font color= 'green' size= 4>  The following cell is only defining the function 'InsideLevels' and will not produce any immediate output. The input is a metadata table and the output consists of a summary dataframe of our metadata. </font></p>

In [ ]:
def InsideLevels(df):
    # get all the columns (equals all attributes) -> will be number of rows
    levels = []
    types = []
    count = []
    for col in df.columns:
        types.append(type(df[col][0]))
        levels.append(sorted(set(df[col].dropna())))
        tmp = df[col].value_counts()
        count.append([tmp[levels[-1][i]] for i in range(len(levels[-1]))])
    return pd.DataFrame({"ATTRIBUTES": df.columns, "LEVELS": levels, "COUNT":count, "TYPES": types}, index=range(1, len(levels)+1))

Let's have a look at our metadata, with the above defined function InsideLevels.

In [ ]:
len(md.columns) #number of columnns of metadata

14

In [ ]:
InsideLevels(md)

,ATTRIBUTES,LEVELS,COUNT,TYPES
1,filename,"[SD_01-2018_10_a.mzXML, SD_01-2018_10_b.mzXML,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<class 'str'>
2,ATTRIBUTE_Sample_Type,"[Blank, Sample]","[6, 180]",<class 'str'>
3,ATTRIBUTE_Batch,"[1, 2, 3]","[62, 62, 62]",<class 'numpy.int64'>
4,ATTRIBUTE_Month,"[Dec, Jan, Oct]","[62, 62, 62]",<class 'str'>
5,ATTRIBUTE_Year,"[2017, 2018]","[62, 124]",<class 'numpy.int64'>
6,ATTRIBUTE_Sample_Location,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.int64'>
7,ATTRIBUTE_Replicate,"[a, b]","[93, 93]",<class 'str'>
8,ATTRIBUTE_Spot,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.int64'>
9,ATTRIBUTE_Latitude,"[32.75645, 32.75743, 32.75905, 32.76115, 32.76...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.float64'>
10,ATTRIBUTE_Longitude,"[-117.2872, -117.28664, -117.286, -117.28355, ...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.float64'>


<p style='text-align: justify;'> The above table is a summary of our metadata tabel. For example, the 2nd row says that there are 2 different types of samples under 'ATTRIBUTE_Sample_Type', namely "Blank" and "Sample". The number of files corresponding to each of these categories is given in the COUNT column. For example, we have 6 files belonging to the "Blank" sample type and 180 files to the "Sample" sample type. </p>

---
## Merging annotations with feature table

### <font color ='darkblue'>  Identifying Appropriate Columns for Merging </font>
    
The first column in feature table: <b>row ID</b> (the unique ID for each detected feature) is given in different columns in different files: <br>
* In GNPS result and GNPS analog result files, the row ID is given in the column: <b>#Scan#</b> ('Compound_Name' column has the annotation information)
* For SIRIUS and CANOPUS summary files, the row ID of the feature table is given in the column <b>id</b>. A typical feature id would be: "3_ProjectName_MZmine3_SIRIUS_1_16", where the last number 16 representing the row ID.

### <font color ='darkblue'> Ensuring Data Compatibility </font>

<p style='text-align: justify;'> Annotating features in a feature table is useful for identifying metabolites that correspond to those features. This can help to understand the biological relevance of the features, thereby assisting in the interpretation of our metabolomics data. Here, we will show how to merge ft and an (analog annotation file) based on the above-mentioned columns. This merged table can be used later, when needed. Before merging two dataframes based on certain columns, make sure that the classes of both columns are the same. Even if the values are the same, but one column is of numeric class and the other of character class, this might cause unwanted errors. </p>

In [ ]:
#checking if both columns are of similar class
ft["row ID"].dtype== an_gnps["#Scan#"].dtype # for feature table and the GNPS result table

True

In [ ]:
ft["row ID"].dtype== an_analog["#Scan#"].dtype # for feature table and the GNPS analog result table

True

### <font color ='darkblue'>  Merging Annotations </font>

<b> Merge the annotations from GNPS: Actual library hits and analogs.</b> This is given in <b>'an_final'</b> dataframe. The compound names of 'an_gnps' and 'an_analog' are combined by a separator ';'. As a result, for each #Scan#, we have a single associated compound name. This result is given in <b>'an_final_single' dataframe.</b>

In [ ]:
# Rename the columns of 'an_analog' with a prefix 'Analog' (excluding the '#Scan#' column)
an_analog.columns = ['Analog_' + col if col != '#Scan#' else col for col in an_analog.columns]

# Merge 'an_analog' with 'an_gnps' using a full join on the '#Scan#' column
an_final = pd.merge(an_gnps, an_analog, on='#Scan#', how='outer')

# Consolidate multiple annotations for a single '#Scan#' into one combined name
def combine_names(row):
    if row['Compound_Name'] == row['Analog_Compound_Name']:
        return row['Compound_Name']
    return ';'.join([str(row['Compound_Name']), str(row['Analog_Compound_Name'])])

an_final_single = an_final.groupby('#Scan#').apply(lambda group: pd.Series({
    'Combined_Name': combine_names(group.iloc[0])
})).reset_index()

# To get the DataFrame with that exact column name (without automatic renaming)
an_final_single.columns = an_final_single.columns.str.replace('.', '_')

In [ ]:
print('Dimension: ', an_final.shape)
an_final.head(n=2)

Dimension:  (1991, 91)


,SpectrumID,Compound_Name,Ion_Source,Instrument,Compound_Source,PI,Data_Collector,Adduct,Precursor_MZ,ExactMass,...,Analog_MoleculeExplorerDatasets,Analog_MoleculeExplorerFiles,Analog_InChIKey,Analog_InChIKey-Planar,Analog_superclass,Analog_class,Analog_subclass,Analog_npclassifier_superclass,Analog_npclassifier_class,Analog_npclassifier_pathway
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,HSHNITRMYYLLCV-UHFFFAOYSA-N,HSHNITRMYYLLCV,NaN,NaN,NaN,Coumarins,Simple coumarins,Shikimates and Phenylpropanoids
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,GOILPRCCOREWQE-UHFFFAOYSA-N,GOILPRCCOREWQE,NaN,NaN,NaN,Pseudoalkaloids,NaN,Alkaloids


In [ ]:
# Save the DataFrame to CSV, including the index (row names) as a column
an_final.to_csv('GNPS_LibraryHits_AnalogHits_Merged.csv', index=True)

In [ ]:
an_final_single # the merged annotation table with the combined names

,#Scan#,Combined_Name
0,2,nan;Massbank:LU096701 4-Methylumbelliferone|7-...
1,11,"nan;""6-methoxypurine, oxamethane CollisionEner..."
2,27,nan;4-((hydroxyimino)methyl)-1-methylhydroquin...
3,83,"nan;""methyl 3-oxo-2-[(3,4,5-trimethoxyphenyl)m..."
4,94,"nan;""3,4-Dihydroxymandelic acid CollisionEnerg..."
...,...,...
1986,92610,nan;4-((hydroxyimino)methyl)-1-methylhydroquin...
1987,92636,nan;HARMANE CollisionEnergy:102040
1988,92638,"""methyl 3-oxo-2-[(3,4,5-trimethoxyphenyl)methy..."
1989,92649,nan;TRYPTOPHAN


This annotation information is then merged with the feature table.

In [ ]:
ft_an = ft.merge(an_final_single, left_on= "row ID",  how='left', right_on= "#Scan#", sort=True)
print('Dimension: ', ft_an.shape)
ft_an.head()

Dimension:  (11217, 202)


,row ID,row m/z,row retention time,row ion mobility,row ion mobility unit,row CCS,correlation group ID,annotation network number,best ion,auto MS2 verify,...,SD_12-2017_27_a.mzXML Peak area,SD_12-2017_29_b.mzXML Peak area,SD_12-2017_21_a.mzXML Peak area,SD_12-2017_30_a.mzXML Peak area,SD_12-2017_28_b.mzXML Peak area,SD_12-2017_29_a.mzXML Peak area,SD_12-2017_28_a.mzXML Peak area,Unnamed: 199,#Scan#,Combined_Name
0,1,391.283813,0.299798,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,0.000,0.0000,0.000,0.000,0.000,0.0000,NaN,NaN,NaN
1,2,161.095931,0.235215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,0.000,0.0000,0.000,0.000,0.000,0.0000,NaN,2.0,nan;Massbank:LU096701 4-Methylumbelliferone|7-...
2,4,167.154202,0.474247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,0.000,0.0000,0.000,0.000,0.000,0.0000,NaN,NaN,NaN
3,5,158.961323,0.477149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23437.6720,22848.000,13511.6570,24267.623,26252.643,23598.620,14783.2270,NaN,NaN,NaN
4,11,172.956270,0.486601,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2476.0715,15583.166,3745.3218,18710.676,19134.986,18255.406,4221.2554,NaN,11.0,"nan;""6-methoxypurine, oxamethane CollisionEner..."


---
## Arranging metadata and feature table in the same order

<p style='text-align: justify;'> In the next cells, we bring feature table and metadata in the correct format such that the rownames of the metadata and column names of the feature table are the same. Filenames and the order of files need to correspond in both tables, as we will match metadata attributes to the feature table. In that way, both metadata and feature table, can easily be filtered. </p>

### <font color ='darkblue'>  Creating Backup Files </font>

In [ ]:
new_md = md.copy() #storing the files under different names to preserve the original files
new_ft = ft.copy()

### <font color ='darkblue'> Cleaning the files </font>

In [ ]:
new_ft.columns = new_ft.columns.str.replace(' Peak area', '') # Removing " Peak area" extensions from the column names of new_ft
new_ft = new_ft.sort_values(by='row ID') # Arranging the rows of new_ft by ascending order of "row ID"

new_ft = new_ft.loc[:, new_ft.notna().sum() > 0] # Removing columns in new_ft where all values are NaN
new_md = new_md.loc[:, new_md.notna().sum() > 0] # Removing columns in new_md where all values are NaN

new_md = new_md[new_md.apply(lambda row: all(item != "" for item in row), axis=1)] # Remove rows where all the elements are empty strings
new_md = new_md.map(lambda x: x.strip() if isinstance(x, str) else x) # Remove leading and trailing spaces from each column of new_md

### <font color ='darkblue'> Update the row names of feature table </font>

In [ ]:
if 'ft_an' in globals():
    comparison_result = (ft_an['row ID'].values == new_ft['row ID'].values).all()
    print(comparison_result)  # Should print True if you have an annotation file

# Changing the index (row names) of new_ft into the combined name as "XID_mz_RT":
new_name = 'X' + new_ft['row ID'].astype(str) + '_' + new_ft['row m/z'].round(3).astype(str) + '_' + new_ft['row retention time'].round(3).astype(str)
new_name_values = new_name.values

if 'ft_an' in globals():
    # Replace NaN values with an empty string before converting to string type
    combined_name_ft = ft_an['Combined_Name'].fillna('').astype(str).values
    underscore_added = ['_' + x for x in combined_name_ft] #add a underscore prefix
    new_name_values = np.core.defchararray.add(new_name_values.astype(str), underscore_added)

# Set the new index and remove trailing underscore if present
new_ft.index = new_name_values

True


In [ ]:
new_ft.head(n=2)

,row ID,row m/z,row retention time,correlation group ID,annotation network number,best ion,identified by n=,partners,neutral M mass,SD_01-2018_5_b.mzXML,...,SD_12-2017_23_b.mzXML,SD_12-2017_15_b.mzXML,SD_12-2017_15_a.mzXML,SD_12-2017_27_a.mzXML,SD_12-2017_29_b.mzXML,SD_12-2017_21_a.mzXML,SD_12-2017_30_a.mzXML,SD_12-2017_28_b.mzXML,SD_12-2017_29_a.mzXML,SD_12-2017_28_a.mzXML
X1_391.284_0.3_,1,391.283813,0.299798,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,2,161.095931,0.235215,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
new_md.head(n=2)

,filename,ATTRIBUTE_Sample_Type,ATTRIBUTE_Batch,ATTRIBUTE_Month,ATTRIBUTE_Year,ATTRIBUTE_Sample_Location,ATTRIBUTE_Replicate,ATTRIBUTE_Spot,ATTRIBUTE_Latitude,ATTRIBUTE_Longitude,ATTRIBUTE_Sample_Area,ATTRIBUTE_Spot_Name,ATTRIBUTE_time_run,ATTRIBUTE_Injection_order
0,SD_10_2018_10_a.mzXML,Sample,3,Oct,2018,10,a,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,18/07/2020 18:19,145
1,SD_10_2018_10_b.mzXML,Sample,3,Oct,2018,10,b,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,18/07/2020 18:35,146


<p style='text-align: justify;'> After adding the annotation information, the feature table "new_ft" now includes the annotation in the row names of the features. This can be beneficial for downstream analysis, such as univariate statistics, where significant features can be easily identified if they are annotated. Similarly, in supervised analysis like Random Forest, the annotated compound driving the classification can be quickly assessed. This will be further highlighted in the respective univariate and multivariate sections. </p>

### <font color ='darkblue'> Selecting Relevant Columns </font>

In [ ]:
# Selecting only the columns with names containing 'mzXML' or 'mzML'
new_ft = new_ft.loc[:, new_ft.columns.str.contains('\\.mzXML$|\\.mzML$')]

# If either .mzXML or .mzML files are present, print a message
if new_ft.columns.str.contains('\\.mzXML$').any() and new_ft.columns.str.contains('\\.mzML$').any():
    print("Both .mzXML and .mzML file types are present in the data")

In [ ]:
# Checking the files again to see if the above changes have been made:
print('Dimension: ', new_ft.shape)
new_ft.head(n=2)

Dimension:  (11217, 186)


,SD_01-2018_5_b.mzXML,SD_01-2018_7_b.mzXML,SD_01-2018_7_a.mzXML,SD_01-2018_3_b.mzXML,SD_01-2018_6_a.mzXML,SD_01-2018_8_a.mzXML,SD_01-2018_1_a.mzXML,SD_01-2018_2_b.mzXML,SD_01-2018_4_b.mzXML,SD_01-2018_2_a.mzXML,...,SD_12-2017_23_b.mzXML,SD_12-2017_15_b.mzXML,SD_12-2017_15_a.mzXML,SD_12-2017_27_a.mzXML,SD_12-2017_29_b.mzXML,SD_12-2017_21_a.mzXML,SD_12-2017_30_a.mzXML,SD_12-2017_28_b.mzXML,SD_12-2017_29_a.mzXML,SD_12-2017_28_a.mzXML
X1_391.284_0.3_,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
print('Dimension: ', new_md.shape)
new_md.head(n=2)

Dimension:  (186, 14)


,filename,ATTRIBUTE_Sample_Type,ATTRIBUTE_Batch,ATTRIBUTE_Month,ATTRIBUTE_Year,ATTRIBUTE_Sample_Location,ATTRIBUTE_Replicate,ATTRIBUTE_Spot,ATTRIBUTE_Latitude,ATTRIBUTE_Longitude,ATTRIBUTE_Sample_Area,ATTRIBUTE_Spot_Name,ATTRIBUTE_time_run,ATTRIBUTE_Injection_order
0,SD_10_2018_10_a.mzXML,Sample,3,Oct,2018,10,a,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,18/07/2020 18:19,145
1,SD_10_2018_10_b.mzXML,Sample,3,Oct,2018,10,b,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,18/07/2020 18:35,146


### <font color ='darkblue'> Verifying File Consistency </font>

In [ ]:
new_ft = new_ft.reindex(columns=sorted(new_ft.columns)) # Ordering the columns of 'new_ft' by their names
new_md = new_md.sort_values(by='filename').reset_index(drop=True) #ordering the md by the 1st column filename

In [ ]:
# how many files in the metadata are also present in the feature table
new_md['filename'].isin(new_ft.columns).value_counts()  #if this returns False it means some files are missing

filename
True    186
Name: count, dtype: int64

The output says that all 186 files are present in both new_md & new_ft. Furthermore, metadata filenames and feature table column names are identical, indicating that they are in the same order. If the above lines returns FALSE, it means some files are missing. To check names of the files that are missing we can run the next cell. If everything went well, the next cell should return the statement that all files are present in both new_md & new_ft.

In [ ]:
# check if new_ft column names and md row names are the same
if sorted(new_ft.columns) == sorted(new_md['filename']):
    print(f"All {len(new_ft.columns)} files are present in both new_md & new_ft.")
else:
    print("Not all files are present in both new_md & new_ft.\n")
    # print the md rows / ft column which are not in ft columns / md rows and remove them
    ft_cols_not_in_md = [col for col in new_ft.columns if col not in new_md['filename']]
    print(f"These {len(ft_cols_not_in_md)} columns of feature table are not present in metadata table and will be removed:\n{', '.join(ft_cols_not_in_md)}\n")
    new_ft.drop(columns=ft_cols_not_in_md, inplace=True)
    md_rows_not_in_ft = [row for row in new_md['filename'] if row not in new_ft.columns]
    print(f"These {len(md_rows_not_in_ft)} rows of metadata table are not present in feature table and will be removed:\n{', '.join(md_rows_not_in_ft)}\n")
    new_md.drop(md_rows_not_in_ft, inplace=True)

All 186 files are present in both new_md & new_ft.


When the above cell returns some filenames, check the corresponding column names in the feature table for spelling mistakes, case-sensitive errors. Reupload the files with correct metadata and rerun the above steps.

In [ ]:
#checking the dimensions of our new ft and md:
print("The number of rows and columns in our original ft is:", ft.shape,"\n")
print("The number of rows and columns in our new ft is:", new_ft.shape,"\n")
print("The number of rows and columns in our new md is:", new_md.shape)

The number of rows and columns in our original ft is: (11217, 200) 

The number of rows and columns in our new ft is: (11217, 186) 

The number of rows and columns in our new md is: (186, 14)


Notice that the number of columns of the new feature table is the same as the number of rows in our new metadata. Now, we have both our feature table and metadata in the same order.

---

# <font color ='blue'> Data-cleanup </font>

### <font color ='darkblue'> Transposing the feature table </font>
As a first step of data-cleanup step, lets merge the metadata and feature table (transposed) together. This can be used later for other purposes such as batch correction.

In [ ]:
ft_t = pd.DataFrame(new_ft).T #transposing the ft
ft_t = ft_t.apply(pd.to_numeric) #converting all values to numeric
np.array_equal(new_md['filename'],ft_t.index) #should return True now

True

In [ ]:
ft_t.head(n=3)

,X1_391.284_0.3_,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,X5_158.961_0.477_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X13_159.065_0.479_,X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,...,X92628_241.068_14.313_,X92636_182.985_14.571_nan;HARMANE CollisionEnergy:102040,"X92638_208.039_14.764_""methyl 3-oxo-2-[(3,4,5-trimethoxyphenyl)methylene]benzo[b]furan-5-carboxylate CollisionEnergy:102040""",X92639_167.013_14.697_,X92640_193.974_14.62_,X92641_194.117_14.485_,X92647_158.961_14.747_,X92648_154.99_14.807_,X92649_226.951_14.813_nan;TRYPTOPHAN,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_10_a.mzXML,0.0,0.0,0.0,23804.152,16765.086,38312.957,0.000,0.0000,0.0,0.0,...,32835.040,74416.95,12208.319,334345.62,28667.012,876652.80,5619.828,3481.2158,0.0,0.000
SD_01-2018_10_b.mzXML,0.0,0.0,0.0,25651.238,18234.424,41744.805,0.000,0.0000,0.0,0.0,...,27659.705,81216.98,21744.803,308244.47,32421.330,908859.06,13885.072,4102.1577,0.0,1016.183
SD_01-2018_11_a.mzXML,0.0,0.0,0.0,25477.540,14966.582,38488.465,6803.451,6466.3687,0.0,0.0,...,30535.060,75824.02,14548.440,317442.66,30157.980,973026.40,19566.814,2840.4290,0.0,0.000


In [ ]:
#merging metadata (new_md) and transposed feature table based on the sample names
ft_merged_with_md = pd.merge(new_md, ft_t.reset_index(), left_on='filename', right_on='index', how='left')
ft_merged_with_md.head(n=2)

,filename,ATTRIBUTE_Sample_Type,ATTRIBUTE_Batch,ATTRIBUTE_Month,ATTRIBUTE_Year,ATTRIBUTE_Sample_Location,ATTRIBUTE_Replicate,ATTRIBUTE_Spot,ATTRIBUTE_Latitude,ATTRIBUTE_Longitude,...,X92628_241.068_14.313_,X92636_182.985_14.571_nan;HARMANE CollisionEnergy:102040,"X92638_208.039_14.764_""methyl 3-oxo-2-[(3,4,5-trimethoxyphenyl)methylene]benzo[b]furan-5-carboxylate CollisionEnergy:102040""",X92639_167.013_14.697_,X92640_193.974_14.62_,X92641_194.117_14.485_,X92647_158.961_14.747_,X92648_154.99_14.807_,X92649_226.951_14.813_nan;TRYPTOPHAN,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
0,SD_01-2018_10_a.mzXML,Sample,2,Jan,2018,10,a,10,32.86261,-117.26042,...,32835.040,74416.95,12208.319,334345.62,28667.012,876652.80,5619.828,3481.2158,0.0,0.000
1,SD_01-2018_10_b.mzXML,Sample,2,Jan,2018,10,b,10,32.86261,-117.26042,...,27659.705,81216.98,21744.803,308244.47,32421.330,908859.06,13885.072,4102.1577,0.0,1016.183


In [ ]:
ft_merged_with_md.to_csv("Ft_md_merged.csv") # This file can be used for batch correction

<div class="alert alert-block alert-warning">
<b><font size=3> Skip the Batch correction section if you do not have multiple batches !! </font> </b> </div>

## Batch Correction (Optional)
<a id="batch_corr"></a>

<p style='text-align: justify;'> A 'Batch' is a group of samples processed and analyzed by the same experimental & instrumental conditions in the same short time period. In general, if we have more samples than the tray size, we might measure them as multiple batches or groups. When arranging samples in a batch for measurement, in order to ensure biological diversity within a batch, in addition to our samples of interest, it is advised to have QCs, blanks, and controls (Wehrens et al., 2016). To merge data from these different batches, we must look for batch-effects, both, between the batches and within each batch and correct these effects. <b>But, prior to batch correction on a dataset, we should evaluate the severity of the batch effect and when it is small, it is best to not perform batch correction as this may result in an incorrect estimation of the biological variance in the data. Instead, we should treat the statistical results with caution (Nygaard et al., 2016). For more details, please read the manuscript </b>.</p>


<p style='text-align: justify;'> In this tutorial, the test dataset was utilized to evaluate the chemical impacts of a significant rain event that occurred in northern San Diego, California (USA) during the Winter of 2017/2018. Despite the presence of a "ATTRIBUTE_Batch" column in the metadata, the 3 groups mentioned are not considered as batches due to their distinct collection conditions. The "ATTRIBUTE_time_run" column clearly indicates that the seawater samples were collected and measured at different times during Dec 2017, Jan 2018 (after rainfall), and Oct 2018, respectively. Also, they were collected 'before' and 'after' rainfall. Therefore, searching for inter-batch effects is not meaningful in our example dataset. In terms of intra-batch effect, since the sample dataset does not have QCs, we cannot correct for the intra-batch effect.</p>

Follow the R notebook for Batch Correction: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Functional-Metabolomics-Lab/FBMN-STATS/blob/main/R/Additional_Notebooks/Batch_Correction.ipynb)

---

## Blank removal
<a id="blank_rem"></a>

<p style='text-align: justify;'> Blank samples contain no analytes of interest and consist, for example, of the solvent, matrix, tissue or growth media that was used to prepare or dissolve the samples and analytes of interest. Such as the analytes, the mass spectral features deriving from blank samples are also detected by the LC-MS/MS instrument.
We need to remove these blank features to obtain accurate results.</p>

<p style='text-align: justify;'>To eliminate these blank features, we initially split the feature table into blanks and samples, and then employ a cutoff filter. Next, we compute the average feature intensities for the blanks and samples, and subsequently calculate the ratio of average_feature_blanks to average_feature_sample. We compare this ratio against the user-defined cutoff to determine which features to be removed. When the cutoff is set to 0.3, it implies that for any feature, up to 30% contribution from the blank and 70% from the sample are allowed. Hence any feature with a ratio greater than 0.3 is removed. By using a lower cutoff, such as 10% (0.1), we would demand a greater contribution from the sample (90%) and restrict the blank's contribution to 10%. Raising the cutoff leads to fewer background features being identified and more analyte features being observed. Conversely, lowering the cutoff is more rigorous and leads to the removal of more features.</p>

<font color = "purple"><b> ▲ CRITICAL: </b></font> If you have performed blank removal during pre-processing, simply skip <b>steps 21-23</b>. Before <b>Step 24</b>, assign your feature table and metadata to 'blk_rem' and 'md_Samples' respectively as the following. This step prepares the data for seamless integration into subsequent steps.  
``blk_rem = ft_t.copy()``  
``md_Samples = new_md.copy()``  

### <font color ='darkblue'>  Examine Metadata Attributes </font>

In order to remove blank features from our samples, we first split our feature table into blanks and samples using the metadata and our InsideLevels function created in [step 7](#explore_md).

In [ ]:
InsideLevels(new_md.iloc[:, 1:]) #skipping the 0th column (filename) and looking at the summary table

,ATTRIBUTES,LEVELS,COUNT,TYPES
1,ATTRIBUTE_Sample_Type,"[Blank, Sample]","[6, 180]",<class 'str'>
2,ATTRIBUTE_Batch,"[1, 2, 3]","[62, 62, 62]",<class 'numpy.int64'>
3,ATTRIBUTE_Month,"[Dec, Jan, Oct]","[62, 62, 62]",<class 'str'>
4,ATTRIBUTE_Year,"[2017, 2018]","[62, 124]",<class 'numpy.int64'>
5,ATTRIBUTE_Sample_Location,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.int64'>
6,ATTRIBUTE_Replicate,"[a, b]","[93, 93]",<class 'str'>
7,ATTRIBUTE_Spot,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.int64'>
8,ATTRIBUTE_Latitude,"[32.75645, 32.75743, 32.75905, 32.76115, 32.76...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.float64'>
9,ATTRIBUTE_Longitude,"[-117.2872, -117.28664, -117.286, -117.28355, ...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...",<class 'numpy.float64'>
10,ATTRIBUTE_Sample_Area,"[Blank, La_Jolla Reefs, La_Jolla_Cove, Mission...","[6, 36, 12, 36, 18, 12, 18, 48]",<class 'str'>


### <font color ='darkblue'>  Separate Sample and Blank Files </font>
<font color="red" size=4> For your metadata summary above, enter the index number for your 'sample-type' column as well as the index numbers for the blank and sample. </font>

In [ ]:
sample_index = 1
sample_attribute = new_md.columns[sample_index]

# Removed interactive input and directly set blank_nums and sample_nums
# df = pd.DataFrame({"LEVELS": InsideLevels(new_md.iloc[:, 1:]).iloc[sample_index-1]["LEVELS"]})
# df.index = [*range(1, len(df)+1)]
# display(df)

# Input for blanks, allowing multiple indices
# blank_indices = input('Enter the index numbers of the blanks separated by commas: ').split(',')
# blank_indices = [int(index.strip()) - 1 for index in blank_indices]
blank_nums = [InsideLevels(new_md.iloc[:,1:]).iloc[sample_index-1]['LEVELS'][0]] # Assuming 'Blank' is at index 0

# Input for samples, allowing multiple indices
# sample_indices = input('Enter the index numbers of the samples separated by commas: ').split(',')
# sample_indices = [int(index.strip()) - 1 for index in sample_indices]
sample_nums = [InsideLevels(new_md.iloc[:,1:]).iloc[sample_index-1]['LEVELS'][1]] # Assuming 'Sample' is at index 1

In [ ]:
# Filter for blanks
md_Blank = new_md[new_md[sample_attribute].isin(blank_nums)]
Blank = ft_t[ft_t.index.isin(md_Blank['filename'])]

# Filter for samples
md_Samples = new_md[new_md[sample_attribute].isin(sample_nums)]
Samples = ft_t[ft_t.index.isin(md_Samples['filename'])]

In [ ]:
# Display the chosen blanks
print('Dimension: ',Blank.shape)
Blank.head(2)

Dimension:  (6, 11217)


,X1_391.284_0.3_,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,X5_158.961_0.477_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X13_159.065_0.479_,X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,...,X92628_241.068_14.313_,X92636_182.985_14.571_nan;HARMANE CollisionEnergy:102040,"X92638_208.039_14.764_""methyl 3-oxo-2-[(3,4,5-trimethoxyphenyl)methylene]benzo[b]furan-5-carboxylate CollisionEnergy:102040""",X92639_167.013_14.697_,X92640_193.974_14.62_,X92641_194.117_14.485_,X92647_158.961_14.747_,X92648_154.99_14.807_,X92649_226.951_14.813_nan;TRYPTOPHAN,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_PPL_Bl_1.mzXML,0.0,0.0,0.0,44974.22,0.0,13554.206,0.0,0.0,0.0,0.0,...,44710.844,129935.94,19431.857,378558.56,50910.260,764071.25,32411.197,23549.158,0.0,0.0
SD_01-2018_PPL_Bl_2.mzXML,0.0,0.0,0.0,42645.43,0.0,41131.316,0.0,0.0,0.0,0.0,...,45302.043,127591.73,15539.985,355748.06,53279.316,713951.10,28731.709,18313.002,0.0,0.0


In [ ]:
# Display the chosen samples
print('Dimension: ',Samples.shape)
Samples.head(2)

Dimension:  (180, 11217)


,X1_391.284_0.3_,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,X5_158.961_0.477_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X13_159.065_0.479_,X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,...,X92628_241.068_14.313_,X92636_182.985_14.571_nan;HARMANE CollisionEnergy:102040,"X92638_208.039_14.764_""methyl 3-oxo-2-[(3,4,5-trimethoxyphenyl)methylene]benzo[b]furan-5-carboxylate CollisionEnergy:102040""",X92639_167.013_14.697_,X92640_193.974_14.62_,X92641_194.117_14.485_,X92647_158.961_14.747_,X92648_154.99_14.807_,X92649_226.951_14.813_nan;TRYPTOPHAN,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_10_a.mzXML,0.0,0.0,0.0,23804.152,16765.086,38312.957,0.0,0.0,0.0,0.0,...,32835.040,74416.95,12208.319,334345.62,28667.012,876652.80,5619.828,3481.2158,0.0,0.000
SD_01-2018_10_b.mzXML,0.0,0.0,0.0,25651.238,18234.424,41744.805,0.0,0.0,0.0,0.0,...,27659.705,81216.98,21744.803,308244.47,32421.330,908859.06,13885.072,4102.1577,0.0,1016.183


### <font color ='darkblue'> Define Cutoff for Blank Features Removal </font>

Now that we have our feature table split into blanks and samples, we can start removing blank features. **<font color='red'> We use a cutoff of 0.3 </font>**, meaning that in order for a feature to be considered of interest, it needs to have a ratio of average_feature_blanks vs average_feature_sample <30%. </font> </br>
<font color='red' size = 4> When you run the cell below, you can interactively change the threshold to any value between 0.1 and 1. </font>

In [ ]:
Cutoff = 0.3

### <font color ='darkblue'> Perform Blank Removal </font>

In [ ]:
# Getting mean for every feature in blank and Samples in a DataFrame named 'Avg_ft'
Avg_ft = pd.DataFrame({'Avg_blank': Blank.mean(axis=0, skipna=False)}) # Set skipna=False to check if there are NA values
Avg_ft['Avg_samples'] = Samples.mean(axis=0, skipna=False) # Adding another column 'Avg_samples' for feature means of samples

# Getting the ratio of blank vs Sample
Avg_ft['Ratio_blank_Sample'] = (Avg_ft['Avg_blank'] + 1) / (Avg_ft['Avg_samples'] + 1)

# Creating a bin with 1s when the ratio > Cutoff, else put 0s
Avg_ft['Bg_bin'] = (Avg_ft['Ratio_blank_Sample'] > Cutoff).astype(int)

# Calculating the number of background features and features present
print("Total no.of features:", Avg_ft.shape[0])
print("No.of Background or noise features:", Avg_ft['Bg_bin'].sum())
print("No.of features after excluding noise:", (Samples.shape[1] - Avg_ft['Bg_bin'].sum()))

# Merging Samples with Avg_ft and selecting only the required rows and columns
blk_rem = pd.concat([Samples.T, Avg_ft], axis=1, join='inner')
blk_rem = blk_rem[blk_rem['Bg_bin'] == 0]  # Picking only the features
blk_rem = blk_rem.drop(columns=['Avg_blank', 'Avg_samples', 'Ratio_blank_Sample', 'Bg_bin'])  # Removing the last 4 columns
blk_rem = blk_rem.T

Total no.of features: 11217
No.of Background or noise features: 2125
No.of features after excluding noise: 9092



**Review the blank removed table**:

In [ ]:
print('Dimension: ',blk_rem.shape)
blk_rem.head(n=3)

Dimension:  (180, 9092)


,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,X21_534.268_0.503_,X35_192.062_0.547_,X39_153.033_0.55_,...,X92572_151.035_13.364_,X92576_311.185_13.406_nan;2'-Hydroxy-a-naphthoflavone CollisionEnergy:102040,"X92582_172.956_13.395_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X92588_294.206_13.508_,X92596_322.237_13.562_,X92603_322.237_13.768_,X92606_172.956_13.801_,X92610_184.985_13.892_nan;4-((hydroxyimino)methyl)-1-methylhydroquinolin-2-one CollisionEnergy:102040,X92619_264.144_14.118_,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_10_a.mzXML,0.0,0.0,16765.086,0.000,0.0000,0.0,0.0,0.0,93114.13,513062.94,...,0.0,0.0,38393.008,10275.198,17839.9880,17766.7050,14290.5500,0.0,111653.445,0.000
SD_01-2018_10_b.mzXML,0.0,0.0,18234.424,0.000,0.0000,0.0,0.0,0.0,136021.73,634986.60,...,0.0,0.0,35004.344,24718.518,7944.9316,14684.2030,16327.9795,0.0,108555.586,1016.183
SD_01-2018_11_a.mzXML,0.0,0.0,14966.582,6803.451,6466.3687,0.0,0.0,0.0,81821.54,425097.75,...,0.0,0.0,14035.124,15673.984,1175.2463,986.2982,7382.6543,0.0,112394.090,0.000


In [ ]:
# metadata without the blanks info:
print('Dimension: ', md_Samples.shape)
md_Samples.head(n=3)

Dimension:  (180, 14)


,filename,ATTRIBUTE_Sample_Type,ATTRIBUTE_Batch,ATTRIBUTE_Month,ATTRIBUTE_Year,ATTRIBUTE_Sample_Location,ATTRIBUTE_Replicate,ATTRIBUTE_Spot,ATTRIBUTE_Latitude,ATTRIBUTE_Longitude,ATTRIBUTE_Sample_Area,ATTRIBUTE_Spot_Name,ATTRIBUTE_time_run,ATTRIBUTE_Injection_order
0,SD_01-2018_10_a.mzXML,Sample,2,Jan,2018,10,a,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,16/01/2018 16:23,83
1,SD_01-2018_10_b.mzXML,Sample,2,Jan,2018,10,b,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,16/01/2018 16:39,84
2,SD_01-2018_11_a.mzXML,Sample,2,Jan,2018,11,a,11,32.85601,-117.26253,SIO_La_Jolla_Shores,La_Jolla_Shores,16/01/2018 16:55,85


In [ ]:
blk_rem.to_csv("Blanks_Removed_with_cutoff_"+ str(Cutoff) + ".csv")

---
## Imputation
<a name = 'imp'></a>

<p style='text-align: justify;'> For several reasons, real world datasets might have some missing values in it, in the form of NA, NANs or 0s. Eventhough the gapfilling step of MZmine fills the missing values, we still end up with some missing values or 0s in our feature table. This could be problematic for statistical analysis. We cannot simply discard those rows or columns with missing values as we will lose valuable data. Instead we can try imputing or replacing the missing values in the data with a meaningful, reasonable guess. Here, first, we use the blank-removed feature table to assess frequencies across relative intensities. The plot from the below cell shows us how many features have which relative intensities. We then create random values between 0 and the minimum value in our blank-removed table and randomly replace all 0s with these random values. </p>

### <font color ='darkblue'> Analyzing the frequency distribution of relative intensities </font>
<a name="imp_freq"></a>

In [ ]:
!pip install -U kaleido

In [ ]:
bins, bins_label, a = [-1, 0, 1, 10], ["-1","0", "1", "10"], 2

while a<=10:
    bins_label.append(np.format_float_scientific(10**a))
    bins.append(10**a)
    a+=1

freq_table = pd.DataFrame(bins_label)
frequency = pd.DataFrame(np.array(np.unique(np.digitize(blk_rem.T.to_numpy(), bins, right=True), return_counts=True)).T).set_index(0)
freq_table = pd.concat([freq_table,frequency], axis=1).fillna(0).drop(0)
freq_table.columns = ["intensity", "Frequency"]
freq_table["Log(Frequency)"] = np.log(freq_table["Frequency"]+1)

# get the lowest intensity (that is not zero) as a cutoff LOD value
cutoff_LOD = round(blk_rem.replace(0, np.nan).min(numeric_only=True).min())

fig = px.bar(freq_table, x="intensity", y="Log(Frequency)", template="plotly_white",  width=600, height=400)

fig.update_traces(marker_color="#696880")
fig.update_layout(font={"color":"grey", "size":12, "family":"Sans"},
                  title={"text":"Frequency plot - Gap Filled", "x":0.5, "font_color":"#3E3D53"},
                  xaxis_title= "Range")

#fig.write_image("frequency_plot.png")

fig.show()

The above histogram shows that, in our, blank-removed feature table, there are many zeros present. And no values in the range between 0 to 1E2. The minimum value greater than 0 in our dataframe is in between 1E2 & 1E3 (and that value is 892).

### <font color ='darkblue'> Step Random Value Generation & Zero Replacement </font>

In [ ]:
# Set the seed for random number generation
np.random.seed(141222)

imp = blk_rem.copy()
imp = imp.map(lambda x: np.random.randint(1, cutoff_LOD) if x == 0 else x)

In [ ]:
print('Dimension: ', imp.shape)
imp.head(n=3)

Dimension:  (180, 9092)


,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,X21_534.268_0.503_,X35_192.062_0.547_,X39_153.033_0.55_,...,X92572_151.035_13.364_,X92576_311.185_13.406_nan;2'-Hydroxy-a-naphthoflavone CollisionEnergy:102040,"X92582_172.956_13.395_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X92588_294.206_13.508_,X92596_322.237_13.562_,X92603_322.237_13.768_,X92606_172.956_13.801_,X92610_184.985_13.892_nan;4-((hydroxyimino)methyl)-1-methylhydroquinolin-2-one CollisionEnergy:102040,X92619_264.144_14.118_,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_10_a.mzXML,201.0,835.0,16765.086,596.000,524.0000,723.0,485.0,274.0,93114.13,513062.94,...,748.0,857.0,38393.008,10275.198,17839.9880,17766.7050,14290.5500,864.0,111653.445,614.000
SD_01-2018_10_b.mzXML,453.0,825.0,18234.424,466.000,801.0000,580.0,130.0,206.0,136021.73,634986.60,...,885.0,208.0,35004.344,24718.518,7944.9316,14684.2030,16327.9795,655.0,108555.586,1016.183
SD_01-2018_11_a.mzXML,671.0,813.0,14966.582,6803.451,6466.3687,192.0,527.0,143.0,81821.54,425097.75,...,498.0,527.0,14035.124,15673.984,1175.2463,986.2982,7382.6543,585.0,112394.090,543.000


In [ ]:
(imp == 0).sum().sum() # checking if there are any zeros in our imputed table

np.int64(0)

In [ ]:
imp.to_csv('Imputed_QuantTable.csv') # save to file

## Normalization
<a id="norm"></a>

Normalization is performed to compensate for differences in total metabolite concentrations among samples (Y. Wu & Li,2016). Many normalization techniques can also correct the batch effects, such as those caused by sample pipetting or extraction. Here, we present 2 types of normalization: Total Ion Current (TIC) or (Probabilistic Quotient Normalization) PQN.

### <font color='darkblue'>Install the package (already addressed)</font>
This step has been omitted in the Python Notebook as all required packages are installed at the beginning.

### <font color ='darkblue'> Total Ion Current (TIC) or sample-centric normalization </font>
<a name="tic"></a>

<p style='text-align: justify;'> TIC is a simple normalization technique that is commonly used as it is easy to implement and computationally inexpensive. It scales the intensities of all features in a sample by the total ion current (or sum) of the sample. TIC normalization is appropriate when the volume of sample injected into the instrument is consistent across all samples, and when the differences in the total ion count among samples are mainly due to differences in the concentration of sample injected. However, TIC has its limitation such as few features with high ion observation can heavily influence the calculation. Also it assumes most metabolites are stable and equally up/down-regulated among samples, which does not hold true in cases like normal vs cancerous tissues comparison. </p>

In [ ]:
normalized = imp.copy()

# Dividing each element of a particular row (as each row is the sample) with its row sum
norm_TIC = normalized.apply(lambda x: x/np.sum(x), axis=1)

print('No NA values in Normalized data:',norm_TIC.isnull().values.any() == False)
print('Dimension: ', norm_TIC.shape)
norm_TIC.head(n=3)

No NA values in Normalized data: True
Dimension:  (180, 9092)


,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,X21_534.268_0.503_,X35_192.062_0.547_,X39_153.033_0.55_,...,X92572_151.035_13.364_,X92576_311.185_13.406_nan;2'-Hydroxy-a-naphthoflavone CollisionEnergy:102040,"X92582_172.956_13.395_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X92588_294.206_13.508_,X92596_322.237_13.562_,X92603_322.237_13.768_,X92606_172.956_13.801_,X92610_184.985_13.892_nan;4-((hydroxyimino)methyl)-1-methylhydroquinolin-2-one CollisionEnergy:102040,X92619_264.144_14.118_,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_10_a.mzXML,1.331646e-07,5.531961e-07,0.000011,3.948562e-07,3.471554e-07,4.789950e-07,3.213175e-07,1.815278e-07,0.000062,0.000340,...,4.955577e-07,5.677714e-07,0.000025,0.000007,1.181918e-05,1.177063e-05,0.000009,5.724089e-07,0.000074,4.067814e-07
SD_01-2018_10_b.mzXML,3.412312e-07,6.214476e-07,0.000014,3.510237e-07,6.033691e-07,4.368965e-07,9.792507e-08,1.551736e-07,0.000102,0.000478,...,6.666438e-07,1.566801e-07,0.000026,0.000019,5.984677e-06,1.106117e-05,0.000012,4.933917e-07,0.000082,7.654599e-07
SD_01-2018_11_a.mzXML,4.812072e-07,5.830424e-07,0.000011,4.879091e-06,4.637352e-06,1.376927e-07,3.779377e-07,1.025524e-07,0.000059,0.000305,...,3.571404e-07,3.779377e-07,0.000010,0.000011,8.428271e-07,7.073231e-07,0.000005,4.195324e-07,0.000081,3.894121e-07


In [ ]:
norm_TIC.to_csv(os.path.join(data_dir, "Normalised_TIC_Quant_table.csv")) # save to file

### <font color ='darkblue'> Probabilistic Quotient Normalization (PQN) (Not available)</font>
<a name="pqn"></a>

This step is only available in the R Notebook and not available in Python Notebook.

---
## Scaling
<a name='scaling'></a>

One can also perform center-scaling after imputation. Scaling is typically done to make sure that the data is centered around 0 and has a consistent spread to adjust for differences in offset between high and low-abundant metabolites, thus leaving only relevant variation for analysis.

In [ ]:
#Setting 'filename' column as 'index' for md_Samples
md_Samples.set_index('filename', inplace=True)
md_Samples.index.name = None

# put the rows in the feature table and metadata in the same order
imp.sort_index(inplace=True)
md_Samples.sort_index(inplace=True)

if (md_Samples.index == imp.index).all():
    pass
else:
    print("WARNING: Sample names in feature and metadata table are NOT the same!")

### <font color ='darkblue'> Center-Scaling </font>

In Python, using the `StandardScaler` class from the `sklearn.preprocessing` module, the user can center and scale the data by subtracting the mean and dividing by the standard deviation (SD).

- The 'fit_transform' function calculates the mean and SD of 'transposed' for later scaling and then transforms 'transposed' based on those statistics. This results in a NumPy array with the scaled values.
- The standardized values are then converted back into a Pandas DataFrame, with the same index and columns as the original 'transposed' DataFrame.
- The 'scaled' DataFrame now contains the same data as 'transposed', but with each feature (column) having a mean of 0 and an SD of 1.

Similar to R, this module allows few operations:

- `with_mean=True` (default): This will center the data by subtracting the mean of each feature. This is equivalent to `center=TRUE` in R's `scale` function.
- `with_std=True` (default): This will scale the data by dividing each feature by its SD. This is equivalent to the case when both `scale=TRUE` and `center=TRUE` are set in R's `scale` function.
- `with_mean=False`: The mean will not be subtracted, but there's no direct equivalent in `StandardScaler` for scaling by the root mean square if centering is not performed

The options in Python's `StandardScaler` are somewhat more limited compared to R's `scale` function.

In [ ]:
# center and scale filtered data
Imp_scaled = pd.DataFrame(StandardScaler().fit_transform(imp),
                      index=imp.index,
                      columns=imp.columns)
Imp_scaled.head(n=2)

,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,X21_534.268_0.503_,X35_192.062_0.547_,X39_153.033_0.55_,...,X92572_151.035_13.364_,X92576_311.185_13.406_nan;2'-Hydroxy-a-naphthoflavone CollisionEnergy:102040,"X92582_172.956_13.395_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X92588_294.206_13.508_,X92596_322.237_13.562_,X92603_322.237_13.768_,X92606_172.956_13.801_,X92610_184.985_13.892_nan;4-((hydroxyimino)methyl)-1-methylhydroquinolin-2-one CollisionEnergy:102040,X92619_264.144_14.118_,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_10_a.mzXML,-0.342852,-0.594422,0.770878,-0.698076,-0.688803,-0.544895,-0.545050,-0.579735,0.964910,0.498183,...,-0.142612,-0.305189,-0.244244,-0.067403,0.700786,0.922444,0.400949,-0.068666,-0.117481,-0.378166
SD_01-2018_10_b.mzXML,-0.327681,-0.594734,0.950660,-0.707899,-0.666675,-0.558067,-0.601093,-0.587369,2.198253,1.300497,...,-0.083043,-0.309351,-0.424738,0.372585,0.057133,0.675513,0.593040,-0.102960,-0.135823,-0.299224


In [ ]:
Imp_scaled.to_csv("Imputed_Scaled_QuantTable.csv") #save the file

### <font color ='darkblue'> Choose data for further analysis </font>

In [ ]:
# Define the dataframe of data choices
cleaned_data_choice = pd.DataFrame({
    'INDEX': range(1, 6),
    'Description': ["Raw data", "Blank removed data", "Imputed data", "TIC Normalized", "Scaled"],
    'Variable_name': ["ft_t", "blk_rem", "imp", "norm_TIC", "Imp_scaled"],
    'metadata_name': ["new_md"] + ["md_Samples"] * 4
})

cleaned_data_choice # Display the data choices to the user

,INDEX,Description,Variable_name,metadata_name
0,1,Raw data,ft_t,new_md
1,2,Blank removed data,blk_rem,md_Samples
2,3,Imputed data,imp,md_Samples
3,4,TIC Normalized,norm_TIC,md_Samples
4,5,Scaled,Imp_scaled,md_Samples


<font color="red" size=4> Enter the index number of the table you want to use for further analysis.</font>

In [ ]:
cleaned_data = globals()['Imp_scaled']
metadata = globals()['md_Samples']

In [ ]:
cleaned_data.head(n=2)

,X2_161.096_0.235_nan;Massbank:LU096701 4-Methylumbelliferone|7-hydroxy-4-methylchromen-2-one,X4_167.154_0.474_,"X11_172.956_0.487_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X15_289.168_0.501_,X16_291.165_0.501_,X17_453.234_0.503_,X19_451.237_0.503_,X21_534.268_0.503_,X35_192.062_0.547_,X39_153.033_0.55_,...,X92572_151.035_13.364_,X92576_311.185_13.406_nan;2'-Hydroxy-a-naphthoflavone CollisionEnergy:102040,"X92582_172.956_13.395_nan;""6-methoxypurine, oxamethane CollisionEnergy:102040""",X92588_294.206_13.508_,X92596_322.237_13.562_,X92603_322.237_13.768_,X92606_172.956_13.801_,X92610_184.985_13.892_nan;4-((hydroxyimino)methyl)-1-methylhydroquinolin-2-one CollisionEnergy:102040,X92619_264.144_14.118_,"X92651_172.956_14.611_nan;""6-methoxypurine, oxamethane CollisionEnergy:205060"""
SD_01-2018_10_a.mzXML,-0.342852,-0.594422,0.770878,-0.698076,-0.688803,-0.544895,-0.545050,-0.579735,0.964910,0.498183,...,-0.142612,-0.305189,-0.244244,-0.067403,0.700786,0.922444,0.400949,-0.068666,-0.117481,-0.378166
SD_01-2018_10_b.mzXML,-0.327681,-0.594734,0.950660,-0.707899,-0.666675,-0.558067,-0.601093,-0.587369,2.198253,1.300497,...,-0.083043,-0.309351,-0.424738,0.372585,0.057133,0.675513,0.593040,-0.102960,-0.135823,-0.299224


In [ ]:
metadata.head(n = 2)

,ATTRIBUTE_Sample_Type,ATTRIBUTE_Batch,ATTRIBUTE_Month,ATTRIBUTE_Year,ATTRIBUTE_Sample_Location,ATTRIBUTE_Replicate,ATTRIBUTE_Spot,ATTRIBUTE_Latitude,ATTRIBUTE_Longitude,ATTRIBUTE_Sample_Area,ATTRIBUTE_Spot_Name,ATTRIBUTE_time_run,ATTRIBUTE_Injection_order
SD_01-2018_10_a.mzXML,Sample,2,Jan,2018,10,a,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,16/01/2018 16:23,83
SD_01-2018_10_b.mzXML,Sample,2,Jan,2018,10,b,10,32.86261,-117.26042,SIO_La_Jolla_Shores,SIO_South_Pier,16/01/2018 16:39,84
